## 기본 실행

### 라이브러리 임포트

In [1]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# CatBoost 설치
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.2 MB/s eta 0:00:00


In [3]:
# 기본
import gc

# 데이터
import numpy as np
import pandas as pd
import cudf

# 모델
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier

# 검증 / 평가
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

# 파일
from google.colab import files

### 기본 설정

In [4]:
# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

### 데이터 로드

In [5]:
!rsync -ah --progress "/content/drive/MyDrive/공모전/Toss/toss/train.parquet" "/content/train.parquet"
!rsync -ah --progress "/content/drive/MyDrive/공모전/Toss/toss/test.parquet"  "/content/test.parquet"

train_data = pd.read_parquet('/content/train.parquet')
test_data  = pd.read_parquet('/content/test.parquet')

sending incremental file list
train.parquet
          8.79G 100%   73.40MB/s    0:01:54 (xfr#1, to-chk=0/1)
sending incremental file list
test.parquet
          1.28G 100%   63.47MB/s    0:00:19 (xfr#1, to-chk=0/1)


## Feature Engineering

### 시간/요일 파생 변수

In [6]:
# hour 정수 변환 함수 정의
def to_hour_int(s):
    return s.astype('string').str.lstrip('0').replace({'': '0'}).astype('int16')

In [7]:
# 시간/요일 파생변수 생성
train_data['hour_int'] = to_hour_int(train_data['hour'])
test_data['hour_int'] = to_hour_int(test_data['hour'])

train_data['day_of_week'] = train_data['day_of_week'].astype('int8')
test_data['day_of_week'] = test_data['day_of_week'].astype('int8')

### SEQ 파생 변수

In [8]:
# seq 파생변수 생성
def gpu_seq_features_inplace_batched(df, seq_col='seq', batch_rows=500000):
    n = len(df)

    arr_len = np.empty(n, dtype=np.int32)
    arr_uniq = np.empty(n, dtype=np.int32)
    arr_div = np.empty(n, dtype=np.float32)

    seq_pd = df[seq_col].astype('string').fillna('')

    for start in range(0, n, batch_rows):
        end = min(start + batch_rows, n)

        s = cudf.Series(seq_pd.iloc[start:end].to_numpy(), dtype='str')

        s = s.str.strip()
        s = s.str.replace(r',+', ',', regex=True)
        s = s.str.strip(',')

        lst = s.str.split(',')

        len_col = lst.list.len().astype('int32')
        uniq_col = lst.list.unique().list.len().astype('int32')

        empties = s.str.len() == 0

        len_col = len_col.where(~empties, 0)
        uniq_col = uniq_col.where(~empties, 0)

        denom = len_col.clip(lower=1)

        div_col = (
            uniq_col.astype('float32')
            / denom.astype('float32')
        ).astype('float32')

        arr_len[start:end] = len_col.to_numpy()
        arr_uniq[start:end] = uniq_col.to_numpy()
        arr_div[start:end] = div_col.to_numpy()

    df['seq_len'] = arr_len
    df['seq_unique'] = arr_uniq
    df['diversity_ratio'] = arr_div


gpu_seq_features_inplace_batched(train_data)
gpu_seq_features_inplace_batched(test_data)

In [9]:
# seq 파생변수 검증
bad_len = (train_data['seq_unique'] > train_data['seq_len']).sum()

bad_div = (
    (train_data['diversity_ratio'] < 0)
    | (train_data['diversity_ratio'] > 1)
).sum()

print(f'uniq > len : {bad_len}')
print(f'diversity_ratio 범위 오류 : {bad_div}')

uniq > len : 0
diversity_ratio 범위 오류 : 0


### 로그 파생 변수

In [10]:
# 로그 파생변수 생성
for df in (train_data, test_data):

    if 'feat_b_3' in df.columns:
        fb3 = pd.to_numeric(df['feat_b_3'], errors='coerce').astype('float32')
        df['feat_b_3_pos_log'] = np.log1p(np.clip(fb3, 0, None)).astype('float32')

    if 'feat_b_5' in df.columns:
        fb5 = pd.to_numeric(df['feat_b_5'], errors='coerce').astype('float32')
        fb5 = np.clip(fb5, None, 1 - 1e-6)
        df['feat_b_5_neglog1p'] = (-np.log1p(-fb5)).astype('float32')

    if 'history_b_3' in df.columns:
        hb3 = pd.to_numeric(df['history_b_3'], errors='coerce').astype('float32')
        df['history_b_3_log1p'] = np.log1p(np.clip(hb3, 0, None)).astype('float32')

    if 'history_b_9' in df.columns:
        hb9 = pd.to_numeric(df['history_b_9'], errors='coerce').astype('float32')
        df['history_b_9_log1p'] = np.log1p(np.clip(hb9, 0, None)).astype('float32')

    if 'history_b_10' in df.columns:
        hb10 = pd.to_numeric(df['history_b_10'], errors='coerce').astype('float32')
        df['history_b_10_log1p'] = np.log1p(np.clip(hb10, 0, None)).astype('float32')

    if 'history_b_30' in df.columns:
        hb30 = pd.to_numeric(df['history_b_30'], errors='coerce').astype('float32')
        df['history_b_30_pos_log'] = np.log1p(np.clip(hb30, 0, None)).astype('float32')

## 모델 입력 구성

### 공통 함수 정의


In [11]:
# TE 함수 정의
def kfold_te(df, y, keys, n_splits=5, alpha=200, seed=42):
    cols = keys if isinstance(keys, (list, tuple)) else [keys]

    global_mean_all = df[y].mean()
    g_all = df.groupby(cols, observed=True)[y].agg(['mean','count']).reset_index()
    g_all['te_all'] = (g_all['count']*g_all['mean'] + alpha*global_mean_all) / (g_all['count'] + alpha)
    g_all = g_all[cols + ['te_all']]

    enc = np.zeros(len(df), dtype='float32')
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in kf.split(df):
        global_mean = df.iloc[tr][y].mean()  # 폴드 내부 평균
        g = df.iloc[tr].groupby(cols, observed=True)[y].agg(['mean','count']).reset_index()
        g['te_fold'] = (g['count']*g['mean'] + alpha*global_mean) / (g['count'] + alpha)

        va_keys = df.iloc[va][cols]
        m = (va_keys
             .merge(g[cols + ['te_fold']], on=cols, how='left')
             .merge(g_all, on=cols, how='left'))  # 폴드에 없는 키 → te_all로 백오프
        te_vals = m['te_fold'].fillna(m['te_all']).fillna(global_mean).astype('float32').values
        enc[va] = te_vals

    dh_map = g_all.rename(columns={'te_all': 'te'})
    return pd.Series(enc, index=df.index, dtype='float32'), dh_map, float(global_mean_all)

# 로그 계산 안정화용
EPS = 1e-7

# 가중치 로그로스 계산 함수 정의
def weighted_logloss(y_true, p, pos_rate=None, eps=EPS):
    if pos_rate is None:
        pos_rate = float(np.mean(y_true))
    p = np.clip(p, eps, 1 - eps)
    w1 = 0.5 / pos_rate
    w0 = 0.5 / (1 - pos_rate)
    return float(-np.mean(np.where(y_true == 1, w1*np.log(p), w0*np.log(1 - p))))

### 학습 컬럼 정리

In [12]:
# 상수 컬럼 제거
drop_cols = ['l_feat_20', 'l_feat_23']

for df in (train_data, test_data):
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# 학습 제외 컬럼
DROP_ALWAYS = ['clicked', 'feat_e_3', 'seq', 'ID']

### Target Encoding

In [13]:
# hour_int 생성
for df in (train_data, test_data):
    if 'hour_int' not in df.columns:
        if 'hour' not in df.columns:
            raise ValueError('hour / hour_int 컬럼 없음')
        df['hour_int'] = pd.to_numeric(df['hour'], errors='coerce').fillna(-1).astype('int16')


# TE 컬럼 확인
train_need_cols = ['day_of_week', 'hour_int', 'gender', 'age_group', 'clicked']
test_need_cols = ['day_of_week', 'hour_int', 'gender', 'age_group']

missing_train = [c for c in train_need_cols if c not in train_data.columns]
missing_test = [c for c in test_need_cols if c not in test_data.columns]

if missing_train:
    raise ValueError(f'train TE 필요 컬럼 없음: {missing_train}')

if missing_test:
    raise ValueError(f'test TE 필요 컬럼 없음: {missing_test}')


# TE 키 타입 정리
type_specs = {
    'day_of_week': 'int16',
    'hour_int': 'int16',
    'gender': 'int8',
    'age_group': 'int8',
}

for df in (train_data, test_data):
    for c, dtype in type_specs.items():
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(-1).astype(dtype)


# TE 생성
te_specs = [
    ('dow_h_te', ['day_of_week', 'hour_int'], 250),
    ('gen_ag_te', ['gender', 'age_group'], 400),
]

for out_col, key_cols, alpha in te_specs:
    tmp_train = train_data[key_cols].copy()
    tmp_train['clicked'] = train_data['clicked'].astype('int8').values

    train_data[out_col], te_map, te_global = kfold_te(
        tmp_train,
        y='clicked',
        keys=key_cols,
        n_splits=5,
        alpha=alpha,
        seed=42
    )

    m = test_data[key_cols].merge(te_map, on=key_cols, how='left')

    train_data[out_col] = train_data[out_col].astype('float32')
    test_data[out_col] = m['te'].fillna(te_global).astype('float32').to_numpy()

    print(f'[TE] {out_col} done')

[TE] dow_h_te done
[TE] gen_ag_te done


### 학습 데이터 생성

In [14]:
# 타깃 분리
y_full = train_data['clicked'].astype('int8')

# 학습 제외 컬럼
drop_cols = list(dict.fromkeys(DROP_ALWAYS + ['clicked']))

# 학습 데이터 구성
X_full = train_data.drop(columns=[c for c in drop_cols if c in train_data.columns])
X_test_holdout = test_data.drop(columns=[c for c in drop_cols if c in test_data.columns])

# test 누락 컬럼 확인
missing_test_cols = [c for c in X_full.columns if c not in X_test_holdout.columns]

if missing_test_cols:
    raise ValueError(f'test 누락 컬럼 있음: {missing_test_cols}')

# test 컬럼 순서 정렬
X_test_holdout = X_test_holdout[X_full.columns]

# TE 컬럼 확인
te_cols = ['dow_h_te', 'gen_ag_te']
missing_te_cols = [c for c in te_cols if c not in X_full.columns or c not in X_test_holdout.columns]

if missing_te_cols:
    raise ValueError(f'TE 컬럼 없음: {missing_te_cols}')

print('[data] X_full:', X_full.shape)
print('[data] X_test_holdout:', X_test_holdout.shape)

[data] X_full: (10704179, 126)
[data] X_test_holdout: (1527298, 126)


### 모델별 입력 변환

In [15]:
# CatBoost 범주형 컬럼
MISSING_TOKEN = '__MISSING__'

target_cat_cols = [
    'gender',
    'age_group',
    'day_of_week',
    'hour',
    'inventory_id',
    'l_feat_4',
    'l_feat_2',
    'feat_b_4',
]

cat_names = [c for c in target_cat_cols if c in X_full.columns]


# 범주형 타입 변환
for c in cat_names:
    train_s = X_full[c].astype('string').fillna(MISSING_TOKEN)
    test_s = X_test_holdout[c].astype('string').fillna(MISSING_TOKEN)

    cat_type = pd.CategoricalDtype(
        categories=pd.unique(pd.concat([train_s, test_s], ignore_index=True)),
        ordered=False
    )

    X_full[c] = train_s.astype(cat_type)
    X_test_holdout[c] = test_s.astype(cat_type)


# 범주형 위치
cat_idx = [X_full.columns.get_loc(c) for c in cat_names]

print(f'[cat] cols={len(cat_names)}')

[cat] cols=8


In [16]:
# XGB 입력 데이터 구성
X_full_tree = X_full.copy()
X_test_tree = X_test_holdout.copy()


# 범주형 정수 인코딩
for c in cat_names:
    X_full_tree[c] = X_full_tree[c].cat.codes.astype('int32')
    X_test_tree[c] = X_test_tree[c].cat.codes.astype('int32')


print('[tree] X_full_tree:', X_full_tree.shape)
print('[tree] X_test_tree:', X_test_tree.shape)

[tree] X_full_tree: (10704179, 126)
[tree] X_test_tree: (1527298, 126)


## 모델 학습 및 제출

### 학습 설정

In [17]:
# fold 수, seed, round scan 범위, test 컬럼 순서를 고정
N_SPLITS = 3
SEED = 42

SCAN_SAMPLE = 500_000
COARSE_POINTS = 12
DENSE_STEP = 25
DENSE_BAND_RATE = 0.10

X_test_cat_fixed = X_test_holdout[X_full.columns]
X_test_tree_fixed = X_test_tree[X_full_tree.columns]

print('[data] X_full:', X_full.shape)
print('[data] X_test_cat_fixed:', X_test_cat_fixed.shape)
print('[data] X_full_tree:', X_full_tree.shape)
print('[data] X_test_tree_fixed:', X_test_tree_fixed.shape)
print('[data] y_full:', y_full.shape)
print('[data] cat_cols:', len(cat_names))

[data] X_full: (10704179, 126)
[data] X_test_cat_fixed: (1527298, 126)
[data] X_full_tree: (10704179, 126)
[data] X_test_tree_fixed: (1527298, 126)
[data] y_full: (10704179,)
[data] cat_cols: 8


### 학습 함수 정의

In [18]:
# 학습 함수 정의
# metric 계산, round scan, fold별 모델 학습 함수를 정의

# 점수 계산 함수 정의
def calc_score(ap, wll):
    return 0.5 * ap + 0.5 * (1.0 / (1.0 + wll))


# 성능 계산 함수 정의
def calc_metrics(y_true, p):
    p = np.clip(p, EPS, 1 - EPS)

    ap = average_precision_score(y_true, p)
    wll = weighted_logloss(y_true, p)

    return {
        'logloss': float(log_loss(y_true, p)),
        'auc': float(roc_auc_score(y_true, p)),
        'ap': float(ap),
        'wll': float(wll),
        'score': float(calc_score(ap, wll))
    }


# 성능 출력 함수 정의
def print_metrics(name, met):
    print(
        f'{name} '
        f'logloss={met["logloss"]:.6f}  '
        f'auc={met["auc"]:.6f}  '
        f'AP={met["ap"]:.6f}  '
        f'WLL={met["wll"]:.6f}  '
        f'score≈{met["score"]:.6f}'
    )


# WLL weight 함수 정의
def make_wll_weight(y):
    y_np = np.asarray(y, dtype=np.int8)

    pos_rate = float(y_np.mean())
    w1 = 0.5 / pos_rate
    w0 = 0.5 / (1 - pos_rate)

    return np.where(y_np == 1, w1, w0).astype(np.float32)


# CatBoost 지정 round 예측 함수 정의
def proba_at_round_cat(model, pool, r):
    p = model.predict(
        pool,
        prediction_type='Probability',
        ntree_end=int(r)
    )[:, 1]

    return p.astype(np.float32)


# scan 인덱스 함수 정의
def make_scan_index(y_va, fold):
    rng = np.random.RandomState(SEED + fold)
    k = min(SCAN_SAMPLE, len(y_va))

    if k >= len(y_va):
        return np.arange(len(y_va))

    y_np = np.asarray(y_va, dtype=np.int8)

    pos_idx = np.flatnonzero(y_np == 1)
    neg_idx = np.flatnonzero(y_np == 0)

    n_pos = max(1, int(round(k * len(pos_idx) / len(y_np))))
    n_pos = min(n_pos, len(pos_idx))
    n_neg = min(k - n_pos, len(neg_idx))

    sub_pos = rng.choice(pos_idx, size=n_pos, replace=False)
    sub_neg = rng.choice(neg_idx, size=n_neg, replace=False)

    sub = np.concatenate([sub_pos, sub_neg])
    rng.shuffle(sub)

    return sub


# CatBoost round scan 함수 정의
def scan_cat_r(model, X_va, y_va, cat_idx, tree_cnt, fold):
    sub = make_scan_index(y_va, fold)

    scan_pool = Pool(X_va.iloc[sub], y_va.iloc[sub], cat_features=cat_idx)
    y_scan = y_va.iloc[sub]

    ntree_start = max(100, int(tree_cnt * 0.30))

    cand = np.unique(
        np.linspace(ntree_start, tree_cnt, num=COARSE_POINTS, dtype=int)
    ).tolist()

    if cand[-1] != tree_cnt:
        cand.append(tree_cnt)

    best_score = -np.inf
    best_wll = float('inf')
    best_ap = -1.0
    r_star = cand[-1]

    for r in cand:
        p = proba_at_round_cat(model, scan_pool, int(r))

        ap = average_precision_score(y_scan, p)
        wll = weighted_logloss(y_scan, p)
        score = calc_score(ap, wll)

        if score > best_score:
            best_score = score
            best_wll = wll
            best_ap = ap
            r_star = int(r)

    band = max(300, int(tree_cnt * DENSE_BAND_RATE))
    lo = max(ntree_start, r_star - band)
    hi = min(tree_cnt, r_star + band)

    for r in range(lo, hi + 1, DENSE_STEP):
        p = proba_at_round_cat(model, scan_pool, int(r))

        ap = average_precision_score(y_scan, p)
        wll = weighted_logloss(y_scan, p)
        score = calc_score(ap, wll)

        if score > best_score:
            best_score = score
            best_wll = wll
            best_ap = ap
            r_star = int(r)

    del scan_pool
    gc.collect()

    return int(r_star), float(best_wll), float(best_ap), float(best_score)


# CatBoost fold 학습 함수 정의
def fit_cat_fold(fold, X_tr, y_tr, X_va, y_va, X_te):
    cat_names_this = [c for c in cat_names if c in X_tr.columns]
    cat_idx = [X_tr.columns.get_loc(c) for c in cat_names_this]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    val_pool = Pool(X_va, y_va, cat_features=cat_idx)
    test_pool = Pool(X_te, cat_features=cat_idx)

    pos_ratio = float(y_tr.mean())
    class_weights = [0.5 / (1 - pos_ratio), 0.5 / pos_ratio]

    model = CatBoostClassifier(
        iterations=6000,
        learning_rate=0.03,
        depth=6,
        leaf_estimation_iterations=3,
        border_count=64,

        random_strength=1.2,
        min_data_in_leaf=192,
        l2_leaf_reg=12,
        one_hot_max_size=1,

        bootstrap_type='Bayesian',
        bagging_temperature=0.9,

        loss_function='Logloss',
        eval_metric='Logloss',

        use_best_model=False,
        verbose=1000,
        metric_period=1000,

        task_type='GPU',
        devices='0',
        class_weights=class_weights,
        random_state=SEED + fold,
        allow_writing_files=False
    )

    print(f'[Fold {fold}][cat] train start')
    model.fit(train_pool, eval_set=val_pool)
    print(f'[Fold {fold}][cat] train done')

    tree_cnt = int(model.tree_count_)

    r_star, scan_wll, scan_ap, scan_score = scan_cat_r(
        model=model,
        X_va=X_va,
        y_va=y_va,
        cat_idx=cat_idx,
        tree_cnt=tree_cnt,
        fold=fold
    )

    print(
        f'[Fold {fold}][cat] '
        f'r*={r_star} / tree_cnt={tree_cnt}  '
        f'scan_WLL={scan_wll:.6f}, '
        f'scan_AP={scan_ap:.6f}, '
        f'scan_score≈{scan_score:.6f}'
    )

    p_va = proba_at_round_cat(model, val_pool, r_star)
    p_te = proba_at_round_cat(model, test_pool, r_star)

    met = calc_metrics(y_va, p_va)

    del train_pool, val_pool, test_pool, model
    gc.collect()

    return p_va, p_te, met, r_star, tree_cnt


# XGBoost 지정 round 예측 함수 정의
def predict_xgb_round(model, X, r):
    p = model.predict_proba(
        X,
        iteration_range=(0, int(r))
    )[:, 1]

    return p.astype(np.float32)


# XGBoost round scan 함수 정의
def scan_xgb_r(model, X_va, y_va, fold):
    sub = make_scan_index(y_va, fold)

    X_scan = X_va.iloc[sub]
    y_scan = y_va.iloc[sub]

    tree_cnt = int(model.get_booster().num_boosted_rounds())
    ntree_start = max(100, int(tree_cnt * 0.30))

    cand = np.unique(
        np.linspace(ntree_start, tree_cnt, num=COARSE_POINTS, dtype=int)
    ).tolist()

    if cand[-1] != tree_cnt:
        cand.append(tree_cnt)

    best_score = -np.inf
    best_wll = float('inf')
    best_ap = -1.0
    r_star = cand[-1]

    for r in cand:
        p = predict_xgb_round(model, X_scan, int(r))

        ap = average_precision_score(y_scan, p)
        wll = weighted_logloss(y_scan, p)
        score = calc_score(ap, wll)

        if score > best_score:
            best_score = score
            best_wll = wll
            best_ap = ap
            r_star = int(r)

    band = max(300, int(tree_cnt * DENSE_BAND_RATE))
    lo = max(ntree_start, r_star - band)
    hi = min(tree_cnt, r_star + band)

    for r in range(lo, hi + 1, DENSE_STEP):
        p = predict_xgb_round(model, X_scan, int(r))

        ap = average_precision_score(y_scan, p)
        wll = weighted_logloss(y_scan, p)
        score = calc_score(ap, wll)

        if score > best_score:
            best_score = score
            best_wll = wll
            best_ap = ap
            r_star = int(r)

    del X_scan
    gc.collect()

    return int(r_star), int(tree_cnt), float(best_wll), float(best_ap), float(best_score)


# XGBoost fold 학습 함수 정의
def fit_xgb_fold(fold, X_tr, y_tr, X_va, y_va, X_te):
    w_tr = make_wll_weight(y_tr)
    w_va = make_wll_weight(y_va)

    model = XGBClassifier(
        n_estimators=4000,
        learning_rate=0.035,
        max_depth=6,
        min_child_weight=128,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_alpha=0.0,
        reg_lambda=12.0,

        objective='binary:logistic',
        eval_metric='logloss',

        tree_method='hist',
        device='cuda',

        base_score=0.5,

        random_state=SEED + fold,
        n_jobs=-1
    )

    print(f'[Fold {fold}][xgb] train start')

    model.fit(
        X_tr,
        y_tr,
        sample_weight=w_tr,
        eval_set=[(X_va, y_va)],
        sample_weight_eval_set=[w_va],
        verbose=500
    )

    print(f'[Fold {fold}][xgb] train done')

    r_star, tree_cnt, scan_wll, scan_ap, scan_score = scan_xgb_r(
        model=model,
        X_va=X_va,
        y_va=y_va,
        fold=fold
    )

    print(
        f'[Fold {fold}][xgb] '
        f'r*={r_star} / tree_cnt={tree_cnt}  '
        f'scan_WLL={scan_wll:.6f}, '
        f'scan_AP={scan_ap:.6f}, '
        f'scan_score≈{scan_score:.6f}'
    )

    p_va = predict_xgb_round(model, X_va, r_star)
    p_te = predict_xgb_round(model, X_te, r_star)

    met = calc_metrics(y_va, p_va)

    del model, w_tr, w_va
    gc.collect()

    return p_va, p_te, met, r_star, tree_cnt

### 3-fold 학습

In [19]:
# 학습
# 3-Fold로 CatBoost, XGBoost 학습후 OOF/test 예측을 생성
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

oof_cat = np.zeros(len(X_full), dtype=np.float32)
oof_xgb = np.zeros(len(X_full), dtype=np.float32)

test_cat = np.zeros((len(X_test_cat_fixed), N_SPLITS), dtype=np.float32)
test_xgb = np.zeros((len(X_test_tree_fixed), N_SPLITS), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
    print('\n' + '=' * 70)
    print(f'[Fold {fold}/{N_SPLITS}]')
    print('=' * 70)

    y_tr = y_full.iloc[tr_idx]
    y_va = y_full.iloc[va_idx]

    # CatBoost 학습
    X_tr_cat = X_full.iloc[tr_idx]
    X_va_cat = X_full.iloc[va_idx]

    p_va, p_te, met, r_star, tree_cnt = fit_cat_fold(
        fold=fold,
        X_tr=X_tr_cat,
        y_tr=y_tr,
        X_va=X_va_cat,
        y_va=y_va,
        X_te=X_test_cat_fixed
    )

    oof_cat[va_idx] = p_va
    test_cat[:, fold - 1] = p_te

    print_metrics(f'[Fold {fold}][cat]', met)

    del X_tr_cat, X_va_cat, p_va, p_te
    gc.collect()

    # XGBoost 학습
    X_tr_tree = X_full_tree.iloc[tr_idx]
    X_va_tree = X_full_tree.iloc[va_idx]

    p_va, p_te, met, r_star, tree_cnt = fit_xgb_fold(
        fold=fold,
        X_tr=X_tr_tree,
        y_tr=y_tr,
        X_va=X_va_tree,
        y_va=y_va,
        X_te=X_test_tree_fixed
    )

    oof_xgb[va_idx] = p_va
    test_xgb[:, fold - 1] = p_te

    print_metrics(f'[Fold {fold}][xgb]', met)

    del X_tr_tree, X_va_tree, y_tr, y_va, p_va, p_te
    gc.collect()


[Fold 1/3]
[Fold 1][cat] train start
0:	learn: 0.6891470	test: 0.6891536	best: 0.6891536 (0)	total: 346ms	remaining: 34m 33s
1000:	learn: 0.5933315	test: 0.5979564	best: 0.5979564 (1000)	total: 2m 34s	remaining: 12m 51s
2000:	learn: 0.5864160	test: 0.5962413	best: 0.5962413 (2000)	total: 5m 6s	remaining: 10m 11s
3000:	learn: 0.5807277	test: 0.5956578	best: 0.5956578 (3000)	total: 7m 37s	remaining: 7m 37s
4000:	learn: 0.5755188	test: 0.5954793	best: 0.5954793 (4000)	total: 10m 8s	remaining: 5m 4s
5000:	learn: 0.5705465	test: 0.5954888	best: 0.5954793 (4000)	total: 12m 39s	remaining: 2m 31s
5999:	learn: 0.5658485	test: 0.5955679	best: 0.5954793 (4000)	total: 15m 11s	remaining: 0us
bestTest = 0.5954792993
bestIteration = 4000
[Fold 1][cat] train done
[Fold 1][cat] r*=5850 / tree_cnt=6000  scan_WLL=0.597370, scan_AP=0.077824, scan_score≈0.351927
[Fold 1][cat] logloss=0.568867  auc=0.743961  AP=0.080780  WLL=0.595556  score≈0.353761
[Fold 1][xgb] train start
[0]	validation_0-logloss:0.6897

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [00:43:24] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[Fold 1][xgb] r*=1709 / tree_cnt=4000  scan_WLL=0.598718, scan_AP=0.080342, scan_score≈0.352922
[Fold 1][xgb] logloss=0.558158  auc=0.742734  AP=0.082033  WLL=0.596990  score≈0.354106

[Fold 2/3]
[Fold 2][cat] train start
0:	learn: 0.6892313	test: 0.6892184	best: 0.6892184 (0)	total: 177ms	remaining: 17m 43s
1000:	learn: 0.5932183	test: 0.5976553	best: 0.5976553 (1000)	total: 2m 33s	remaining: 12m 46s
2000:	learn: 0.5864157	test: 0.5960931	best: 0.5960931 (2000)	total: 5m 4s	remaining: 10m 9s
3000:	learn: 0.5807940	test: 0.5954918	best: 0.5954918 (3000)	total: 7m 36s	remaining: 7m 36s
4000:	learn: 0.5756027	test: 0.5952479	best: 0.5952479 (4000)	total: 10m 7s	remaining: 5m 3s
5000:	learn: 0.5707001	test: 0.5951925	best: 0.5951925 (5000)	total: 12m 38s	remaining: 2m 31s
5999:	learn: 0.5660025	test: 0.5952335	best: 0.5951925 (5000)	total: 15m 9s	remaining: 0us
bestTest = 0.595192526
bestIteration = 5000
[Fold 2][cat] train done
[Fold 2][cat] r*=6000 / tree_cnt=6000  scan_WLL=0.594473, sc

### 앙상블 제출 생성

In [20]:
# 예측 및 제출 생성
# OOF 기준 최적 앙상블 weight를 선택하고 제출 csv를 생성

test_cat_mean = test_cat.mean(axis=1).astype(np.float32)
test_xgb_mean = test_xgb.mean(axis=1).astype(np.float32)

best_score = -np.inf
best_w = None
best_met = None

# 앙상블 weight 탐색
for w_cat in np.round(np.arange(0, 1.01, 0.05), 2):
    w_xgb = round(1.0 - w_cat, 2)

    p_oof = np.clip(
        w_cat * oof_cat + w_xgb * oof_xgb,
        EPS,
        1 - EPS
    )

    met = calc_metrics(y_full, p_oof)

    if met['score'] > best_score:
        best_score = met['score']
        best_w = (float(w_cat), float(w_xgb))
        best_met = met

w_cat, w_xgb = best_w

print('\n[best ensemble]')
print(f'cat={w_cat:.2f}, xgb={w_xgb:.2f}')
print_metrics('[ensemble OOF]', best_met)


# 최종 test 예측
test_pred_final = np.clip(
    w_cat * test_cat_mean + w_xgb * test_xgb_mean,
    EPS,
    1 - EPS
).astype(np.float32)

print('\n[final test]')
print(f'mean={test_pred_final.mean():.8f}')
print(f'min={test_pred_final.min():.8f}')
print(f'max={test_pred_final.max():.8f}')


# 제출 파일 생성
sample_submission = pd.read_csv('/content/drive/MyDrive/공모전/Toss/toss/sample_submission.csv')

submission = sample_submission.copy()
submission['clicked'] = test_pred_final

submission_path = f'submission_cat{w_cat:.2f}_xgb{w_xgb:.2f}.csv'
submission.to_csv(submission_path, index=False)

print('\n[save]', submission_path)
print(submission.head())

files.download(submission_path)


[best ensemble]
cat=0.45, xgb=0.55
[ensemble OOF] logloss=0.563830  auc=0.744294  AP=0.082538  WLL=0.595307  score≈0.354688

[final test]
mean=0.39806521
min=0.02104425
max=0.98837280

[save] submission_cat0.45_xgb0.55.csv
             ID   clicked
0  TEST_0000000  0.292889
1  TEST_0000001  0.328280
2  TEST_0000002  0.399082
3  TEST_0000003  0.501600
4  TEST_0000004  0.251748


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>